In [1]:
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -r requirements.txt

Cloning into 'yolov5'...
remote: Enumerating objects: 17851, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 17851 (delta 33), reused 11 (delta 11), pack-reused 17805 (from 2)
Receiving objects: 100% (17851/17851), 16.97 MiB | 8.37 MiB/s, done.
Resolving deltas: 100% (12166/12166), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 15.1 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [2]:
!ls

benchmarks.py	 data	     LICENSE	     README.zh-CN.md   tutorial.ipynb
CITATION.cff	 detect.py   models	     requirements.txt  utils
classify	 export.py   pyproject.toml  segment	       val.py
CONTRIBUTING.md  hubconf.py  README.md	     train.py


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!mkdir -p /content/drive/MyDrive/ai-infrastructure-inspection

In [5]:
!nvidia-smi

Tue Mar 31 11:54:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             16W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
!pip install kagglehub

In [7]:
import kagglehub

path = kagglehub.dataset_download("aliabdelmenam/rdd-2022")

print("Dataset path:", path)

100%|██████████| 9.90G/9.90G [10:55<00:00, 16.2MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/aliabdelmenam/rdd-2022/versions/1


In [9]:
print(path)

/root/.cache/kagglehub/datasets/aliabdelmenam/rdd-2022/versions/1


In [10]:
!ls /root/.cache/kagglehub/datasets/aliabdelmenam/rdd-2022/versions/1

RDD_SPLIT


In [20]:
import os
import shutil
from glob import glob

base = "/root/.cache/kagglehub/datasets/aliabdelmenam/rdd-2022/versions/1/RDD_SPLIT"

out = "/content/rdd_small"
os.makedirs(out + "/train/images", exist_ok=True)
os.makedirs(out + "/train/labels", exist_ok=True)
os.makedirs(out + "/val/images", exist_ok=True)
os.makedirs(out + "/val/labels", exist_ok=True)

In [21]:
import random

img_paths = glob(base + "/train/images/*.jpg")
random.shuffle(img_paths)

subset = img_paths[:3000]  # number of images (change anytime)

for img in subset:
    label = img.replace("images", "labels").replace(".jpg", ".txt")

    shutil.copy(img, out + "/train/images/")
    if os.path.exists(label):
        shutil.copy(label, out + "/train/labels/")

In [22]:
val_imgs = glob(base + "/val/images/*.jpg")
val_subset = val_imgs[:500]

for img in val_subset:
    label = img.replace("images", "labels").replace(".jpg", ".txt")

    shutil.copy(img, out + "/val/images/")
    if os.path.exists(label):
        shutil.copy(label, out + "/val/labels/")

In [23]:
%%writefile data.yaml
train: /content/rdd_small/train/images
val: /content/rdd_small/val/images

nc: 5
names: ['longitudinal crack', 'transverse crack', 'alligator crack', 'other corruption', 'pothole']

Overwriting data.yaml


In [24]:
!python train.py --img 640 --batch 32 --epochs 15 --data data.yaml --weights yolov5s.pt --device 0

Streaming output truncated to the last 5000 lines.
       2/14      8.45G    0.07172    0.02235    0.02752         68        640:  12% 11/94 [00:21<03:03,  2.22s/it]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
       2/14      8.45G    0.07158    0.02243    0.02696         88        640:  13% 12/94 [00:21<02:18,  1.69s/it]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
       2/14      8.45G    0.07192     0.0226    0.02704         96        640:  14% 13/94 [00:25<03:04,  2.28s/it]/content/yolov5/train.py:414: FutureW

In [25]:
!python detect.py \
--weights runs/train/exp3/weights/best.pt \
--source /content/rdd_small/val/images \
--img 640 \
--conf 0.25

detect: weights=['runs/train/exp3/weights/best.pt'], source=/content/rdd_small/val/images, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_format=0, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-463-g88af13e3 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 157 layers, 7023610 parameters, 0 gradients, 15.8 GFLOPs
image 1/500 /content/rdd_small/val/images/China_Drone_000000.jpg: 640x640 1 alligator crack, 11.4ms
image 2/500 /content/rdd_small/val/images/China_Drone_000061.jpg: 640x640 1 transverse crack, 1 pothole, 11.4ms
image 3/500 /content/rdd_small/val/images/China_Drone_000144.jpg: 640x640 (no detections),

In [26]:
!zip -r results.zip runs/detect/exp

  adding: runs/detect/exp/ (stored 0%)
  adding: runs/detect/exp/United_States_003141.jpg (deflated 8%)
  adding: runs/detect/exp/Japan_005254.jpg (deflated 5%)
  adding: runs/detect/exp/United_States_004345.jpg (deflated 8%)
  adding: runs/detect/exp/India_005692.jpg (deflated 5%)
  adding: runs/detect/exp/Norway_005228.jpg (deflated 5%)
  adding: runs/detect/exp/Japan_009859.jpg (deflated 4%)
  adding: runs/detect/exp/India_008015.jpg (deflated 5%)
  adding: runs/detect/exp/India_001858.jpg (deflated 6%)
  adding: runs/detect/exp/Norway_000097.jpg (deflated 6%)
  adding: runs/detect/exp/India_000942.jpg (deflated 6%)
  adding: runs/detect/exp/China_Drone_000292.jpg (deflated 7%)
  adding: runs/detect/exp/Japan_002825.jpg (deflated 3%)
  adding: runs/detect/exp/Japan_005616.jpg (deflated 5%)
  adding: runs/detect/exp/India_005014.jpg (deflated 5%)
  adding: runs/detect/exp/Norway_003499.jpg (deflated 5%)
  adding: runs/detect/exp/Norway_006119.jpg (deflated 6%)
  adding: runs/detect/e

In [27]:
from google.colab import files
files.download("results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
!ls runs/train

exp  exp2  exp3


In [33]:
!ls runs/train/exp3/weights

best.pt  last.pt


In [34]:
!cp runs/train/exp3/weights/best.pt /content/best.pt

In [35]:
from google.colab import files
files.download("/content/best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:
!zip -r results_exp3.zip runs/train/exp3

  adding: runs/train/exp3/ (stored 0%)
  adding: runs/train/exp3/train_batch2.jpg (deflated 2%)
  adding: runs/train/exp3/val_batch1_labels.jpg (deflated 18%)
  adding: runs/train/exp3/confusion_matrix.png (deflated 21%)
  adding: runs/train/exp3/val_batch0_labels.jpg (deflated 10%)
  adding: runs/train/exp3/results.png (deflated 7%)
  adding: runs/train/exp3/val_batch2_pred.jpg (deflated 6%)
  adding: runs/train/exp3/P_curve.png (deflated 7%)
  adding: runs/train/exp3/hyp.yaml (deflated 45%)
  adding: runs/train/exp3/events.out.tfevents.1774961684.557aa286dc62.20763.0 (deflated 24%)
  adding: runs/train/exp3/PR_curve.png (deflated 9%)
  adding: runs/train/exp3/train_batch1.jpg (deflated 3%)
  adding: runs/train/exp3/opt.yaml (deflated 50%)
  adding: runs/train/exp3/val_batch1_pred.jpg (deflated 19%)
  adding: runs/train/exp3/results.csv (deflated 81%)
  adding: runs/train/exp3/labels_correlogram.jpg (deflated 32%)
  adding: runs/train/exp3/train_batch0.jpg (deflated 3%)
  adding: runs

In [37]:
from google.colab import files
files.download("results_exp3.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>